# GPT-2 Fine-Tuning — Hands-On Project
### Fine-tuning `gpt2` for Domain-Specific Text Generation (Quotes dataset)

**Goal:** Fine-tune a pretrained GPT-2 (causal language model) so it generates text in the style of a target domain — here, inspirational quotes. The same pipeline works with any plain-text corpus (student essays, poetry, product descriptions, etc.).

**What you'll learn / do:**
1. Load a text dataset (`https://huggingface.co/datasets/Abirate/english_quotes`)
2. Tokenize with GPT-2's BPE tokenizer for causal language modeling
3. Load `GPT2LMHeadModel` and fine-tune with `Trainer`
4. Generate text before vs. after fine-tuning to see the effect

**Runtime:** Go to `Runtime > Change runtime type > GPU (T4)` before starting.

## 1. Install Dependencies

In [1]:
!pip install -q transformers datasets accelerate

## 2. Imports & GPU Check

In [2]:
import torch
from datasets import load_dataset, Dataset
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
if device.type != "cuda":
    print("WARNING: No GPU detected. Go to Runtime > Change runtime type > GPU.")

Using device: cuda


## 3. Load Dataset

**Option A (default, quick demo):** a public quotes dataset (`Abirate/english_quotes`).

**Option B (use your own text):** uncomment the upload cell below to fine-tune on your own `.txt` file (e.g. lecture notes, a book, students' essays) instead.

In [3]:

raw_dataset = load_dataset("Abirate/english_quotes")
texts = raw_dataset["train"]["quote"]
print(f"Loaded {len(texts)} text samples")
print(texts[:3])

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Loaded 2508 text samples
['“Be yourself; everyone else is already taken.”', "“I'm selfish, impatient and a little insecure. I make mistakes, I am out of control and at times hard to handle. But if you can't handle me at my worst, then you sure as hell don't deserve me at my best.”", "“Two things are infinite: the universe and human stupidity; and I'm not sure about the universe.”"]


In [4]:
# --- Option B: use your own uploaded .txt file instead ---
# Uncomment to use. Each line in the file will be treated as one training example.

# from google.colab import files
# uploaded = files.upload()  # choose your .txt file
# fname = list(uploaded.keys())[0]
# with open(fname, "r", encoding="utf-8") as f:
#     texts = [line.strip() for line in f if line.strip()]
# print(f"Loaded {len(texts)} lines from {fname}")

## 4. Build a Hugging Face Dataset & Tokenize

In [5]:
# ------------------------------------------------------------
# Step 1: Select the GPT-2 model
# ------------------------------------------------------------
# You can use:
# "gpt2"         -> Small model (124M parameters)
# "gpt2-medium"  -> Medium model (355M parameters)
# Larger models require more GPU memory and training time.
MODEL_NAME = "gpt2"

# ------------------------------------------------------------
# Step 2: Load the GPT-2 tokenizer
# ------------------------------------------------------------
# The tokenizer converts text into numerical token IDs
# that the GPT-2 model can understand.
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)

# ------------------------------------------------------------
# Step 3: Set the padding token
# ------------------------------------------------------------
# GPT-2 does not have a predefined padding token.
# Therefore, we use the End-of-Sequence (EOS) token
# as the padding token.
tokenizer.pad_token = tokenizer.eos_token

# ------------------------------------------------------------
# Step 4: Create a Hugging Face Dataset
# ------------------------------------------------------------
# Convert the list of text samples into a Dataset object.
# Example:
# texts = [
#     "Machine learning is amazing.",
#     "Deep learning is a subset of AI."
# ]
dataset = Dataset.from_dict({"text": texts})

# ------------------------------------------------------------
# Step 5: Define the tokenization function
# ------------------------------------------------------------
# This function converts raw text into token IDs.
#
# truncation=True
#     Cuts longer sentences to the maximum length.
#
# max_length=128
#     Keeps only the first 128 tokens.
#
# padding="max_length"
#     Pads shorter sentences with the pad token so that
#     every sequence has exactly 128 tokens.
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length"
    )

# ------------------------------------------------------------
# Step 6: Apply tokenization to the entire dataset
# ------------------------------------------------------------
# batched=True
#     Processes multiple samples at once (faster).
#
# remove_columns=["text"]
#     Removes the original text column after tokenization,
#     keeping only the tokenized features.
tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"]
)

# ------------------------------------------------------------
# Step 7: Split the dataset
# ------------------------------------------------------------
# Divide the dataset into:
#   90% Training Data
#   10% Evaluation (Validation) Data
#
# seed=42 ensures the split is reproducible.
split = tokenized_dataset.train_test_split(
    test_size=0.1,
    seed=42
)

# ------------------------------------------------------------
# Step 8: Create training dataset
# ------------------------------------------------------------
train_dataset = split["train"]

# ------------------------------------------------------------
# Step 9: Create evaluation dataset
# ------------------------------------------------------------
eval_dataset = split["test"]

# ------------------------------------------------------------
# Step 10: Display information about the training dataset
# ------------------------------------------------------------
# Shows:
#   • Number of samples
#   • Feature names
#   • Dataset structure
print(train_dataset)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2257
})


## 5. Data Collator for Causal Language Modeling
`mlm=False` tells the collator to do next-token prediction (GPT-style) rather than masked LM (BERT-style).

In [6]:
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 6. See What the Base (Untrained) Model Generates

In [7]:
# ------------------------------------------------------------
# Step 1: Load the pre-trained GPT-2 language model
# ------------------------------------------------------------
# GPT2LMHeadModel is GPT-2 with a language modeling head,
# which is used for predicting the next word (token).
#
# from_pretrained() downloads or loads the pre-trained model.
#
# .to(device) moves the model to the selected device
# (GPU if available, otherwise CPU).
base_model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

# ------------------------------------------------------------
# Step 2: Create a function to generate text
# ------------------------------------------------------------
def generate_text(model, prompt, max_new_tokens=40):

    # --------------------------------------------------------
    # Convert the input prompt into token IDs
    # --------------------------------------------------------
    # return_tensors="pt"
    #     Returns PyTorch tensors instead of Python lists.
    #
    # .to(device)
    #     Moves the input tensors to the same device as the model.
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # --------------------------------------------------------
    # Generate new text using the GPT-2 model
    # --------------------------------------------------------
    output = model.generate(

        # Pass the tokenized input to the model
        **inputs,

        # Maximum number of new words/tokens to generate
        max_new_tokens=max_new_tokens,

        # Enable random sampling instead of greedy decoding
        # This produces more natural and diverse text.
        do_sample=True,

        # Top-K Sampling
        # Select the next token only from the top 50
        # most probable tokens.
        top_k=50,

        # Top-P (Nucleus Sampling)
        # Choose tokens whose cumulative probability
        # reaches 95%.
        top_p=0.95,

        # Temperature controls randomness.
        #
        # temperature < 1.0
        #     More focused and predictable output.
        #
        # temperature = 1.0
        #     Normal randomness.
        #
        # temperature > 1.0
        #     More creative and random output.
        temperature=0.9,

        # GPT-2 has no default padding token.
        # Use the EOS (End-of-Sequence) token as padding.
        pad_token_id=tokenizer.eos_token_id,
    )

    # --------------------------------------------------------
    # Convert generated token IDs back into readable text
    # --------------------------------------------------------
    # skip_special_tokens=True removes special tokens
    # such as <EOS>.
    return tokenizer.decode(
        output[0],
        skip_special_tokens=True
    )

# ------------------------------------------------------------
# Step 3: Define an input prompt
# ------------------------------------------------------------
# GPT-2 will continue writing from this sentence.
prompt = "The secret to success is"

# ------------------------------------------------------------
# Step 4: Generate text BEFORE fine-tuning
# ------------------------------------------------------------
# This allows you to compare the model's responses
# before and after fine-tuning.
print("BEFORE fine-tuning:\n", generate_text(base_model, prompt))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

BEFORE fine-tuning:
 The secret to success is to use the right tools, not just the right stuff. A lot of people fail in their first try. The best advice to the best people is to try everything. If you cannot learn from your


## 7. Load Fresh Model, Set Up Training Arguments & Trainer

In [8]:
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME).to(device)

training_args = TrainingArguments(
    output_dir="./gpt2-finetuned-checkpoints",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=25,
    load_best_model_at_end=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

## 8. Fine-Tune GPT-2

In [9]:
trainer.train()

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.171410,2.918734
2,2.812944,2.921382
3,2.685456,2.940873


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=849, training_loss=2.891053131247297, metrics={'train_runtime': 187.4635, 'train_samples_per_second': 36.119, 'train_steps_per_second': 4.529, 'total_flos': 442302087168000.0, 'train_loss': 2.891053131247297, 'epoch': 3.0})

## 9. Evaluate Perplexity

In [10]:
# ------------------------------------------------------------
# Step 1: Import the math module
# ------------------------------------------------------------
# The math module provides mathematical functions.
# Here, we use math.exp() to calculate perplexity.
import math

# ------------------------------------------------------------
# Step 2: Evaluate the fine-tuned model
# ------------------------------------------------------------
# trainer.evaluate() runs the model on the evaluation
# (validation) dataset and returns various metrics,
# including the evaluation loss.
eval_results = trainer.evaluate()

# ------------------------------------------------------------
# Step 3: Calculate Perplexity
# ------------------------------------------------------------
# Perplexity is a common evaluation metric for
# language models such as GPT.
#
# Formula:
#     Perplexity = e^(Evaluation Loss)
#
# Lower perplexity indicates better model performance.
perplexity = math.exp(eval_results["eval_loss"])

# ------------------------------------------------------------
# Step 4: Display the evaluation loss
# ------------------------------------------------------------
# :.4f formats the loss value to 4 decimal places.
print(f"Eval loss: {eval_results['eval_loss']:.4f}")

# ------------------------------------------------------------
# Step 5: Display the perplexity
# ------------------------------------------------------------
# :.2f formats the perplexity to 2 decimal places.
print(f"Perplexity: {perplexity:.2f}")

Training Loss,Validation Loss,Epoch
2.685456,2.918734,3


Eval loss: 2.9187
Perplexity: 18.52


## 10. Save the Fine-Tuned Model

In [11]:
SAVE_DIR = "./gpt2-finetuned"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

# Optional: zip and download to your machine
# !zip -r gpt2-finetuned.zip ./gpt2-finetuned
# from google.colab import files
# files.download("gpt2-finetuned.zip")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./gpt2-finetuned


## 11. Compare Generation: Before vs. After Fine-Tuning

In [12]:
finetuned_model = GPT2LMHeadModel.from_pretrained(SAVE_DIR).to(device)

prompts = [
    "The secret to success is",
    "Life is like",
    "In the end, we all",
]

for p in prompts:
    print("PROMPT:", p)
    print("  BASE     :", generate_text(base_model, p))
    print("  FINE-TUNED:", generate_text(finetuned_model, p))
    print()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

PROMPT: The secret to success is
  BASE     : The secret to success is not necessarily that you're going to win or that you're going to win against a lot of people, but that you can make them feel comfortable. Sometimes you need to have some sort of psychological response
  FINE-TUNED: The secret to success is to be patient and to take what you have. And if you don't take what you've got, you're in trouble. To listen to what's going on is to be part of what's

PROMPT: Life is like
  BASE     : Life is like the ocean where a man lives. As a man, he'll always have the same thoughts and feelings. He'll always think that there are other things he can do that are not on his mind,
  FINE-TUNED: Life is like the moon, it's an invisible light and it's an easy place to be.” But I'm not sure, and I'm not a scientist.” It's a matter of time

PROMPT: In the end, we all
  BASE     : In the end, we all know how the government works, even if it's a little ridiculous.

The public has no right to know 

## Deployment

## https://dashboard.ngrok.com/get-started/your-authtoken

In [13]:
!pip install streamlit pyngrok -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 94.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 107.9 MB/s eta 0:00:00


In [14]:
%%writefile app.py
import streamlit as st
import torch
import time
import random
from transformers import GPT2Tokenizer, GPT2LMHeadModel
import re

def clean_and_trim(text):
    # Fix common encoding issues
    text = text.replace("â€™", "'").replace("â€œ", '"').replace("â€", '"')
    # Find the last sentence-ending punctuation
    matches = list(re.finditer(r'[.!?]', text))
    if matches:
        last_punct = matches[-1].end()  # position after the last punctuation
        # Only trim if the punctuation is not too early (e.g., after first few words)
        if last_punct > len(text) * 0.4:  # ensure we don't trim to just the first sentence
            text = text[:last_punct]
    return text
# -------------------- Page config --------------------
st.set_page_config(
    page_title="✨ GPT‑2 Quote Studio",
    page_icon="📝",
    layout="wide"
)

# -------------------- Custom CSS for a beautiful UI --------------------
st.markdown("""
    <style>
        /* Animated gradient background */
        .stApp {
            background: linear-gradient(-45deg, #ee7752, #e73c7e, #23a6d5, #23d5ab);
            background-size: 400% 400%;
            animation: gradient 15s ease infinite;
        }
        @keyframes gradient {
            0% { background-position: 0% 50%; }
            50% { background-position: 100% 50%; }
            100% { background-position: 0% 50%; }
        }

        /* Glass-morphism card for output */
        .glass {
            background: rgba(255, 255, 255, 0.15);
            backdrop-filter: blur(20px);
            -webkit-backdrop-filter: blur(20px);
            border-radius: 20px;
            border: 1px solid rgba(255, 255, 255, 0.25);
            box-shadow: 0 8px 32px 0 rgba(31, 38, 135, 0.37);
            padding: 2rem;
            margin: 1rem 0;
        }

        /* Header styling */
        .main-header {
            font-size: 3.5rem;
            font-weight: 800;
            background: linear-gradient(120deg, #fff 20%, #ffe6e6 80%);
            -webkit-background-clip: text;
            -webkit-text-fill-color: transparent;
            text-shadow: 0 4px 20px rgba(0,0,0,0.1);
            text-align: center;
        }

        /* Subtext */
        .sub-header {
            color: rgba(255,255,255,0.85);
            text-align: center;
            font-size: 1.2rem;
            margin-bottom: 2rem;
        }

        /* Generated text inside glass */
        .generated-text {
            font-family: 'Georgia', serif;
            font-size: 1.4rem;
            line-height: 1.8;
            color: #1a1a2e;
            white-space: pre-wrap;
            word-break: break-word;
        }

        /* Custom button */
        .stButton button {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            color: white;
            font-weight: 600;
            border: none;
            border-radius: 50px;
            padding: 0.6rem 2rem;
            transition: transform 0.2s, box-shadow 0.2s;
            box-shadow: 0 4px 15px rgba(0,0,0,0.2);
        }
        .stButton button:hover {
            transform: scale(1.02);
            box-shadow: 0 6px 25px rgba(0,0,0,0.3);
        }

        /* Sidebar styling */
        .css-1d391kg {
            background: rgba(255,255,255,0.1) !important;
            backdrop-filter: blur(10px);
        }
        .sidebar .sidebar-content {
            background: rgba(0,0,0,0.05);
        }
        .stSlider label {
            color: white !important;
            font-weight: 500;
        }
        .stMarkdown {
            color: #f0f0f0;
        }
        .footer {
            margin-top: 3rem;
            color: rgba(255,255,255,0.6);
            text-align: center;
            font-size: 0.9rem;
        }
        /* Random prompt button in sidebar */
        .random-btn {
            background: rgba(255,255,255,0.2) !important;
            border: 1px solid rgba(255,255,255,0.3) !important;
        }
    </style>
""", unsafe_allow_html=True)

# -------------------- Load model (cached) --------------------
@st.cache_resource
def load_model(model_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = GPT2Tokenizer.from_pretrained(model_path)
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_path).to(device)
    return tokenizer, model, device

MODEL_PATH = "./gpt2-finetuned"
try:
    tokenizer, model, device = load_model(MODEL_PATH)
    st.sidebar.success("✅ Loaded fine‑tuned model")
except:
    st.sidebar.warning("⚠️ Using base GPT‑2 (fine‑tuned model not found)")
    tokenizer, model, device = load_model("gpt2")

# -------------------- Sidebar controls --------------------
with st.sidebar:
    st.markdown("## ⚙️ Controls")

    temperature = st.slider("🌡️ Temperature", 0.1, 2.0, 0.85, 0.05,
                            help="Higher → more random, lower → more focused")
    top_k = st.slider("🔝 Top‑k", 1, 100, 50,
                      help="Sample from the top k tokens")
    top_p = st.slider("🎯 Top‑p (nucleus)", 0.0, 1.0, 0.92, 0.05,
                      help="Cumulative probability cutoff")
    max_new_tokens = st.slider("📏 Max new tokens", 10, 200, 60, 5,
                               help="Maximum length of generated text")

    st.markdown("---")
    st.markdown("### 🎲 Random Prompt")
    if st.button("✨ Surprise me!", use_container_width=True, key="random_prompt"):
        # Pre‑defined prompt starters (you can extend this list)
        starters = [
            "The secret to happiness is",
            "Life is like a",
            "True wisdom comes from",
            "In the end, we all",
            "Courage is not the absence of fear, but",
            "The best way to predict the future is",
            "Love is",
            "Success is not about",
            "The only thing we have to fear is",
            "To be or not to be, that is"
        ]
        st.session_state.prompt_text = random.choice(starters)

# -------------------- Main UI --------------------
st.markdown('<div class="main-header">📖 Quote Studio</div>', unsafe_allow_html=True)
st.markdown('<div class="sub-header">✨ Fine‑tuned GPT‑2 that writes elegant quotes</div>', unsafe_allow_html=True)

# Input area
prompt = st.text_area(
    "✍️ Your prompt",
    value=st.session_state.get("prompt_text", "The secret to success is"),
    height=100,
    placeholder="Type your prompt here...",
    key="prompt_input"
)

# Buttons
col1, col2, col3 = st.columns([1, 2, 1])
with col1:
    generate = st.button("🚀 Generate", use_container_width=True)
with col2:
    clear = st.button("🧹 Clear output", use_container_width=True)

# Placeholder for generated text (will be updated in real‑time)
output_placeholder = st.empty()

# -------------------- Generation with typing effect --------------------
def type_effect(text, placeholder, delay=0.02):
    """Simulate typing by displaying text character by character."""
    displayed = ""
    for char in text:
        displayed += char
        placeholder.markdown(f'<div class="glass"><div class="generated-text">{displayed}</div></div>',
                             unsafe_allow_html=True)
        time.sleep(delay)
    return displayed

if generate and prompt:
    with st.spinner("🤖 Thinking..."):
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            output = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                repetition_penalty=1.3,          # slightly stronger
                early_stopping=True
            )
        full_text = tokenizer.decode(output[0], skip_special_tokens=True)
        full_text = clean_and_trim(full_text)    # <-- add this line

    # Display with typing effect...
    # Optional: trim to last sentence (if you want)
    # Uncomment the following lines to auto‑trim at a sentence boundary
    # import re
    # match = re.search(r'([.!?])\s*$', full_text)
    # if not match:
    #     last_punct = max(full_text.rfind('.'), full_text.rfind('!'), full_text.rfind('?'))
    #     if last_punct > len(full_text)//2:
    #         full_text = full_text[:last_punct+1]

    # Real‑time typing effect
    typed = type_effect(full_text, output_placeholder, delay=0.025)

    # Show continuation separately
    if full_text.startswith(prompt):
        continuation = full_text[len(prompt):].lstrip()
        st.markdown(f"**✏️ Continuation:** _{continuation}_")

elif clear:
    output_placeholder.empty()

# If no generation, show a default message
if not generate and not clear:
    output_placeholder.markdown(
        '<div class="glass" style="text-align:center; color:#888;">'
        '✨ Your generated quote will appear here with a live typing effect.'
        '</div>', unsafe_allow_html=True
    )

st.markdown('<div class="footer">Fine‑tuned on English quotes · Built with ❤️ using Streamlit</div>', unsafe_allow_html=True)

Writing app.py


In [18]:
import os
import subprocess
import time
from pyngrok import ngrok

# Kill any existing ngrok processes (frees up the 3-tunnel limit)
ngrok.kill()
# Set your real token (get from dashboard)
ngrok.set_auth_token("3GDRBYyQJAOmvXg4glZEp2oaVkK_4BRFq4CRbuaB7Sg8Mk2sT")    # <-- replace this

# Kill previous Streamlit
!pkill -f streamlit || true

# Start Streamlit
subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8500"])
time.sleep(5)   # wait for server to start

# Expose port
public_url = ngrok.connect(8500)
print(f"🌍 Your Streamlit app is live at: {public_url}")

^C
🌍 Your Streamlit app is live at: NgrokTunnel: "https://flatworm-banked-diminish.ngrok-free.dev" -> "http://localhost:8500"


In [16]:
ngrok.kill()